# E0 — MLP trên landmark thô (khung cố định index=22), có flip augmentation + TTA
Nhóm 9 — VSL 34 lớp. Đánh giá cross-subject (LOSO): train trên 3 người, test trên người còn lại, xoay vòng 4 lần.

Giao thức thống nhất với E1 (chỉ khác bước chuẩn hóa tọa độ):
- MLP 63 → 128 → 64 → 34, ReLU, Dropout 0.3; Adam lr=1e-3; batch 32; tối đa 100 epoch; ReduceLROnPlateau (factor 0.5, patience 15, min_lr 1e-5).
- Validation 15% tách phân tầng theo lớp từ 3 người train; seed = 42 + fold.
- Flip augmentation: tách train/val TRƯỚC, sau đó nhân đôi tập fit = bản gốc + bản lật ngang. Validation giữ mẫu gốc.
- Lật ngang trên tọa độ ảnh thô của MediaPipe: x' = 1 − x. Tay/khung mất landmark (toàn 0) giữ nguyên 0.
- TTA khi test: trung bình **softmax** của bản gốc và bản lật, rồi argmax.
- Latency: CPU, 1 mẫu, gồm cả 2 lượt forward của TTA; 10 lượt khởi động, trung bình 100 lần đo.

## 1. Import + cấu hình

In [ ]:
import re
import json
import time
from pathlib import Path
from typing import Protocol

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang chạy trên: {DEVICE}")

# Dataset: https://www.kaggle.com/datasets/hauuto/vietnamese-sign-language-alphabet
# Tự dò thư mục landmarks/landmarks/raw vì Kaggle có thể mount dataset ở nhiều đường dẫn khác nhau.
_candidates = list(Path("/kaggle/input").rglob("landmarks/landmarks/raw"))
if not _candidates:
    raise FileNotFoundError(
        "Không tìm thấy landmarks/landmarks/raw trong /kaggle/input — "
        "kiểm tra lại đã Add Data đúng dataset vietnamese-sign-language-alphabet chưa."
    )
LANDMARK_DIR = _candidates[0]
print(f"Dùng LANDMARK_DIR = {LANDMARK_DIR}")

PEOPLE = ("hau", "khoi", "tai", "vy")
FNAME_RE = re.compile(r"^([a-z_]+)_([a-z]+)_([AB])_(\d+)\.npy$")

CHECKPOINT_DIR = Path("/kaggle/working/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR = Path("/kaggle/working/plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)
RESULT_PATH = Path("/kaggle/working/E0_results.json")

FRAME_INDEX = 22        # khung cố định thứ 23/45, dùng chung cho E0/E1
EPOCHS = 100
LEARNING_RATE = 1e-3
BATCH_SIZE = 32
VAL_RATIO = 0.15
SEED = 42               # seed thực tế của mỗi fold = SEED + fold_index (giống E1)
VALID_EPS = 1e-7        # ngưỡng coi một tay là "có landmark" (tổng |giá trị| > VALID_EPS)

LATENCY_WARMUP = 10
LATENCY_RUNS = 100

MAU_XANH = "#2a78d6"
MAU_CAM = "#eb6834"

## 2. Đọc dữ liệu landmark

In [ ]:
def load_all_landmarks():
    records = []
    for f in sorted(LANDMARK_DIR.glob("*/*.npy")):
        m = FNAME_RE.match(f.name)
        if not m:
            print(f"CẢNH BÁO: tên file không đúng quy ước: {f.name}")
            continue
        code, person, block, seq = m.groups()
        arr = np.load(f)
        records.append({"code": code, "person": person, "block": block, "seq": int(seq), "arr": arr})
    return records


records = load_all_landmarks()
print(f"Tổng số mẫu đọc được: {len(records)}")

ALL_CLASSES = sorted(set(r["code"] for r in records))
CODE_TO_IDX = {c: i for i, c in enumerate(ALL_CLASSES)}
IDX_TO_CODE = {i: c for c, i in CODE_TO_IDX.items()}
print(f"Số lớp: {len(ALL_CLASSES)}")

## 3. Khung đánh giá cross-subject (dùng chung cho cả 4 thực nghiệm)

In [ ]:
class ModelStrategy(Protocol):
    def prepare_input(self, X: np.ndarray) -> np.ndarray: ...
    def train(self, X_train, y_train, tag: str): ...
    def predict(self, model_state, X_test) -> np.ndarray: ...
    def measure_latency(self, model_state, X_sample) -> float: ...


def run_cross_subject(records, strategy: ModelStrategy, people=PEOPLE):
    results = []
    for test_person in people:
        train_records = [r for r in records if r["person"] != test_person]
        test_records = [r for r in records if r["person"] == test_person]

        X_train_raw = np.stack([r["arr"] for r in train_records])
        y_train = np.array([CODE_TO_IDX[r["code"]] for r in train_records])
        X_test_raw = np.stack([r["arr"] for r in test_records])
        y_test = np.array([CODE_TO_IDX[r["code"]] for r in test_records])

        X_train = strategy.prepare_input(X_train_raw)
        X_test = strategy.prepare_input(X_test_raw)

        model_state = strategy.train(X_train, y_train, tag=test_person)
        y_pred = strategy.predict(model_state, X_test)
        accuracy = float(np.mean(y_pred == y_test))
        latency_ms = strategy.measure_latency(model_state, X_test[:1])

        results.append({"test_person": test_person, "accuracy": accuracy, "latency_ms": latency_ms})
        print(f"Test trên {test_person}: accuracy={accuracy:.3f}, latency={latency_ms:.3f}ms (gồm 2 forward TTA)")

    accs = [r["accuracy"] for r in results]
    lats = [r["latency_ms"] for r in results]
    print(f"\nTrung bình: accuracy={np.mean(accs):.3f} (±{np.std(accs):.3f}), "
          f"latency={np.mean(lats):.3f}ms (±{np.std(lats):.3f}ms)")
    return results

## 4. Hàm vẽ biểu đồ train loss / val accuracy

In [ ]:
def plot_history(history, tag: str, fold_idx: int, exp: str = "E0"):
    fig, ax1 = plt.subplots(figsize=(7, 4.5))

    ax1.plot(history["epoch"], history["train_loss"], color=MAU_XANH, linewidth=2, label="Train loss")
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Train loss", color=MAU_XANH)
    ax1.tick_params(axis="y", labelcolor=MAU_XANH)

    ax2 = ax1.twinx()
    ax2.plot(history["epoch"], history["val_acc"], color=MAU_CAM, linewidth=2, label="Val accuracy")
    ax2.set_ylabel("Val accuracy", color=MAU_CAM)
    ax2.tick_params(axis="y", labelcolor=MAU_CAM)
    ax2.set_ylim(0, 1)

    fig.suptitle(f"{exp} — Train loss & Val accuracy (fold test={tag})")
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc="center right")

    fig.tight_layout()
    out_path = PLOT_DIR / f"{exp}_fold{fold_idx}_{tag}.png"
    fig.savefig(out_path, dpi=150)
    plt.close(fig)
    print(f"  Đã lưu biểu đồ: {out_path}")

## 5. Kiến trúc MLP (giống hệt E1)

In [ ]:
class MLPClassifier(nn.Module):
    def __init__(self, in_dim: int, num_classes: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        return self.net(x)

## 6. Strategy cho E0

In [ ]:
class E0Strategy:
    """MLP trên landmark THÔ tại khung index=22. Không chuẩn hóa tọa độ (khác biệt duy nhất so với E1)."""

    def __init__(self):
        self.fold_index = 0

    def prepare_input(self, X: np.ndarray) -> np.ndarray:
        # (N, 45, 63|126) -> (N, 63|126): lấy đúng khung cố định, giữ nguyên tọa độ ảnh của MediaPipe.
        return X[:, FRAME_INDEX, :].astype(np.float32, copy=True)

    @staticmethod
    def flip_horizontal(X: np.ndarray) -> np.ndarray:
        """Lật ngang landmark ở tọa độ ảnh thô: x' = 1 - x.

        - Chỉ lật tay có landmark; tay mất landmark (toàn 0) giữ nguyên 0,
          tránh biến 0 thành 1 và tạo ra điểm giả.
        - Với 2 tay (126 chiều) thì đổi chỗ 2 block để slot trái/phải vẫn đúng sau khi lật.
        """
        if X.ndim != 2 or X.shape[1] not in (63, 126):
            raise ValueError(f"Expected landmark vectors (N, 63/126), got {X.shape}")
        hands = X.reshape(X.shape[0], -1, 21, 3).copy()
        valid = np.abs(hands).sum(axis=(-1, -2)) > VALID_EPS          # (N, n_hands)
        hands[..., 0] = np.where(valid[..., None], 1.0 - hands[..., 0], hands[..., 0])
        if hands.shape[1] == 2:
            hands = hands[:, ::-1].copy()
        return hands.reshape(X.shape).astype(np.float32)

    @staticmethod
    def split_train_validation(y: np.ndarray, seed: int):
        """Chia train/validation phân tầng theo lớp, chỉ từ 3 người train (giống E1)."""
        rng = np.random.default_rng(seed)
        train_indices, validation_indices = [], []
        for class_index in np.unique(y):
            indices = np.flatnonzero(y == class_index)
            rng.shuffle(indices)
            if len(indices) < 2:
                raise ValueError(f"Lớp {class_index} có ít hơn 2 mẫu trong tập train")
            validation_count = min(max(1, int(len(indices) * VAL_RATIO)), len(indices) - 1)
            validation_indices.extend(indices[:validation_count])
            train_indices.extend(indices[validation_count:])
        return np.asarray(train_indices), np.asarray(validation_indices)

    def train(self, X_train, y_train, tag: str):
        fold_seed = SEED + self.fold_index
        np.random.seed(fold_seed)
        torch.manual_seed(fold_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(fold_seed)

        # Tách validation TRƯỚC khi lật, để bản lật của mẫu validation không lọt vào tập fit.
        train_idx, val_idx = self.split_train_validation(y_train, fold_seed)
        X_fit, y_fit = X_train[train_idx], y_train[train_idx]
        X_val, y_val = X_train[val_idx], y_train[val_idx]

        # Flip augmentation cố định: mỗi mẫu fit có đúng 1 bản gốc + 1 bản lật.
        X_fit = np.concatenate((X_fit, self.flip_horizontal(X_fit)), axis=0)
        y_fit = np.concatenate((y_fit, y_fit), axis=0)

        model = MLPClassifier(X_fit.shape[1], len(ALL_CLASSES)).to(DEVICE)
        optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode="max", factor=0.5, patience=15, min_lr=1e-5
        )
        criterion = nn.CrossEntropyLoss()

        generator = torch.Generator().manual_seed(fold_seed)
        loader = DataLoader(
            TensorDataset(torch.from_numpy(X_fit).float(), torch.from_numpy(y_fit).long()),
            batch_size=BATCH_SIZE, shuffle=True, generator=generator,
        )
        X_val_t = torch.from_numpy(X_val).float().to(DEVICE)
        y_val_t = torch.from_numpy(y_val).long().to(DEVICE)

        history = {"epoch": [], "train_loss": [], "val_acc": []}
        best_val_acc, best_epoch, best_state = -1.0, 0, None

        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0.0
            for xb, yb in loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(model(xb), yb)
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * xb.size(0)
            train_loss = total_loss / len(X_fit)

            # Validation chỉ trên mẫu gốc, không TTA (giống E1) — dùng để chọn checkpoint.
            model.eval()
            with torch.inference_mode():
                val_acc = float((model(X_val_t).argmax(dim=1) == y_val_t).float().mean().item())
            scheduler.step(val_acc)

            history["epoch"].append(epoch + 1)
            history["train_loss"].append(train_loss)
            history["val_acc"].append(val_acc)

            if val_acc > best_val_acc:
                best_val_acc, best_epoch = val_acc, epoch + 1
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

            if (epoch + 1) % 20 == 0:
                lr_now = optimizer.param_groups[0]["lr"]
                print(f"  [E0 fold {self.fold_index + 1}/4] epoch {epoch + 1}/{EPOCHS} "
                      f"loss={train_loss:.4f} val_acc={val_acc:.3f} lr={lr_now:.2e}")

        model.load_state_dict(best_state)
        model.eval()
        print(f"  Chọn checkpoint tốt nhất tại epoch {best_epoch}/{EPOCHS} (val_acc={best_val_acc:.3f})")

        ckpt_path = CHECKPOINT_DIR / f"E0_fold{self.fold_index}_{tag}.pt"
        torch.save({
            "experiment": "E0",
            "test_person": tag,
            "frame_index": FRAME_INDEX,
            "coordinate_normalization": False,
            "horizontal_flip_train": True,
            "horizontal_flip_test_tta": True,
            "tta_merge": "mean_softmax",
            "input_dim": int(X_fit.shape[1]),
            "classes": ALL_CLASSES,
            "best_epoch": best_epoch,
            "best_val_accuracy": best_val_acc,
            "model_state_dict": model.state_dict(),
        }, ckpt_path)
        print(f"  Đã lưu checkpoint: {ckpt_path}")

        plot_history(history, tag, self.fold_index)
        self.fold_index += 1
        return model

    def _tta_probs(self, model, X: np.ndarray, device) -> torch.Tensor:
        """Trung bình softmax của bản gốc và bản lật (quy ước TTA chung của nhóm)."""
        original = torch.from_numpy(X).float().to(device)
        flipped = torch.from_numpy(self.flip_horizontal(X)).float().to(device)
        return (torch.softmax(model(original), dim=1) + torch.softmax(model(flipped), dim=1)) / 2

    def predict(self, model_state, X_test) -> np.ndarray:
        model_state.eval()
        with torch.inference_mode():
            return self._tta_probs(model_state, X_test, DEVICE).argmax(dim=1).cpu().numpy()

    def measure_latency(self, model_state, X_sample) -> float:
        """Đo trên CPU, 1 mẫu, gồm lật + 2 forward + gộp softmax (toàn bộ bước TTA)."""
        original_device = next(model_state.parameters()).device
        model_state.to("cpu").eval()
        with torch.inference_mode():
            for _ in range(LATENCY_WARMUP):
                self._tta_probs(model_state, X_sample, "cpu")
            elapsed_ms = []
            for _ in range(LATENCY_RUNS):
                start = time.perf_counter()
                self._tta_probs(model_state, X_sample, "cpu")
                elapsed_ms.append((time.perf_counter() - start) * 1000.0)
        model_state.to(original_device)
        return float(np.mean(elapsed_ms))

## 7. Kiểm tra nhanh phép lật (chạy trước khi train)

In [ ]:
_s = E0Strategy()
_X = _s.prepare_input(np.stack([r["arr"] for r in records[:8]]))
_Xf = _s.flip_horizontal(_X)
_hands = _X.reshape(_X.shape[0], -1, 21, 3)
_valid = np.abs(_hands).sum(axis=(-1, -2)) > VALID_EPS

print("Shape trước/sau prepare_input:", np.stack([r["arr"] for r in records[:8]]).shape, "->", _X.shape)
assert np.allclose(_s.flip_horizontal(_Xf), _X, atol=1e-6), "Lật 2 lần (sai số float32) phải ra lại dữ liệu gốc"
assert np.allclose(_Xf.reshape(_hands.shape)[..., 1:], _hands[..., 1:]), "y, z không được thay đổi"
if (~_valid).any():
    assert np.all(_Xf.reshape(_hands.shape)[~_valid] == 0), "Tay mất landmark phải giữ nguyên 0"
assert np.isfinite(_Xf).all()
print("Kiểm tra phép lật: OK")

## 8. Chạy đánh giá cross-subject

In [ ]:
strategy = E0Strategy()
results = run_cross_subject(records, strategy)

with open(RESULT_PATH, "w", encoding="utf-8") as f:
    json.dump({
        "experiment": "E0",
        "description": "MLP tren landmark tho, khung co dinh index=22, khong chuan hoa toa do, "
                       "flip augmentation (1-x) + TTA trung binh softmax",
        "protocol": {
            "frame_index": FRAME_INDEX,
            "val_ratio": VAL_RATIO,
            "seed": f"{SEED} + fold_index",
            "horizontal_flip_train": "deterministic pair (original + mirrored), split before flip",
            "tta": "mean softmax of original and mirrored",
            "latency": f"CPU, batch 1, 2 forwards (TTA), warmup {LATENCY_WARMUP}, mean of {LATENCY_RUNS}",
        },
        "results": results,
        "accuracy_mean": float(np.mean([r["accuracy"] for r in results])),
        "accuracy_std": float(np.std([r["accuracy"] for r in results])),
        "latency_mean_ms": float(np.mean([r["latency_ms"] for r in results])),
        "latency_std_ms": float(np.std([r["latency_ms"] for r in results])),
    }, f, ensure_ascii=False, indent=2)

print(f"\nĐã lưu kết quả vào: {RESULT_PATH}")
print("Cần tải về: checkpoints/E0_fold*.pt, plots/E0_fold*.png, E0_results.json, "
      "và notebook này (nhớ lưu kèm output).")